# PyCaret Time Series API for Bitcoin Forecasting

This notebook demonstrates real-time Bitcoin price forecasting using:
- Live data from CoinGecko API
- PyCaret's time series module
- Data pipeline building.

**References**:
- [CoinGecko API Docs](https://www.coingecko.com/en/api/documentation)
- [PyCaret Time Series Docs](https://pycaret.gitbook.io/docs/get-started/modules)



In [1]:
#import os
#os.environ['LD_PRELOAD'] = '/lib/aarch64-linux-gnu/libgomp.so.1'  # For ARM systems
# OR for regular Linux:
#os.environ['LD_PRELOAD'] = '/usr/lib/x86_64-linux-gnu/libgomp.so.1'

# Must be set BEFORE importing pycaret/
#os.environ['LD_PRELOAD'] = '/usr/lib/aarch64-linux-gnu/libgomp.so.1'

#from pycaret.time_series import *


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Imports
import logging
import requests
import numpy as np
import pandas as pd
import plotly.express as px
from pycaret.time_series import *
from datetime import datetime, timedelta

# Configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
pd.set_option('display.max_columns', 50)


In [2]:
import pycaret
print(pycaret.__version__)

3.3.2


In [3]:
class CoinGeckoAPI:
    """
    Professional API client for CoinGecko with error handling and rate limiting.
    """
    
    BASE_URL = "https://api.coingecko.com/api/v3"
    
    def __init__(self, api_key=None):
        self.session = requests.Session()
        self.api_key = api_key
        self.last_call = None
        self.RATE_LIMIT = 10  # calls per minute
        
    def get_ohlc(self, coin_id="bitcoin", vs_currency="usd", days="max"):
        """
        Get OHLC data with professional error handling.
        
        :param coin_id: Cryptocurrency ID (default: bitcoin)
        :param vs_currency: Target currency (default: usd)
        :param days: Data history length (default: max)
        :return: DataFrame with columns [timestamp, open, high, low, close]
        """
        endpoint = f"{self.BASE_URL}/coins/{coin_id}/ohlc"
        params = {
            "vs_currency": vs_currency,
            "days": days
        }
        
        try:
            response = self.session.get(endpoint, params=params)
            response.raise_for_status()
            data = response.json()
            
            df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            df.set_index('timestamp', inplace=True)
            
            logger.info(f"Successfully fetched {len(df)} records")
            return df
            
        except Exception as e:
            logger.error(f"API Error: {str(e)}")
            raise

# Initialize client
cg = CoinGeckoAPI()

In [4]:
def fetch_and_validate_data(days=90):
    """
    End-to-end data pipeline with quality checks and feature engineering.
    
    Parameters:
    -----------
    days : int
        Number of days of historical data to fetch
        
    Returns:
    --------
    pd.DataFrame
        Processed data ready for analysis
    """
    # Fetch raw data
    raw_data = cg.get_ohlc(days=days)
    
    # Validation Checks
    assert isinstance(raw_data, pd.DataFrame), "Data must be a DataFrame"
    assert not raw_data.empty, "Empty data received"
    assert {'open', 'high', 'low', 'close'}.issubset(raw_data.columns), "Missing OHLC columns"
    
    # Data Cleaning
    data = raw_data.copy()
    data = data[~data.index.duplicated(keep='first')]
    
    # Feature Engineering
    data['daily_return'] = data['close'].pct_change()
    data['volatility_7d'] = data['daily_return'].rolling(7).std()
    data['volume_ema_14'] = data['close'].ewm(span=14).mean()
    
    logger.info(f"Data validation passed. Final shape: {data.shape}")
    return data

# Fetch and display data
btc_data = fetch_and_validate_data()
display(btc_data.tail(3))

INFO:__main__:Successfully fetched 23 records
INFO:__main__:Data validation passed. Final shape: (23, 7)


,open,high,low,close,daily_return,volatility_7d,volume_ema_14
timestamp,,,,,,,
2025-04-30,94739.0,95491.0,92953.0,94256.0,-0.005455,0.034763,87978.831674
2025-05-04,94275.0,97822.0,93334.0,95923.0,0.017686,0.031335,89085.564536
2025-05-08,95893.0,97512.0,93592.0,97026.0,0.011499,0.028500,90185.200939


In [5]:
def setup_experiment(data):
    """
    Properly configured PyCaret time series experiment for financial data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Must have datetime index and numeric columns
        
    Returns:
    --------
    TSForecastingExperiment
    """
    try:
        # Ensure proper data formatting
        if not isinstance(data.index, pd.DatetimeIndex):
            raise ValueError("Data must have datetime index")
            
        # Convert to numeric and drop NA
        numeric_data = data.select_dtypes(include=np.number)
        numeric_data = numeric_data.dropna()
        
        
        # Initialize experiment
        exp = setup(
            data=numeric_data,
            target='close',
            #index=data.index,
            fold_strategy='expanding',
            fold=3,
            numeric_imputation_target='ffill',
            session_id=42,
            fh=7,
            verbose=True
        )
        logger.info("PyCaret experiment successfully initialized")
        return exp
        
    except Exception as e:
        logger.error(f"Setup failed: {str(e)}")
        raise

# Properly formatted data
btc_data_formatted = btc_data[['close']].copy()
btc_data_formatted = btc_data_formatted.asfreq('D').ffill()  # Ensure daily frequency

# Initialize experiment
exp = setup_experiment(btc_data_formatted)

,Description,Value
0,session_id,42
1,Target,close
2,Approach,Univariate
3,Exogenous Variables,Not Present
4,Original data shape,"(89, 1)"
5,Transformed data shape,"(89, 1)"
6,Transformed train set shape,"(82, 1)"
7,Transformed test set shape,"(7, 1)"
8,Rows with missing values,0.0%
9,Fold Generator,ExpandingWindowSplitter


INFO:__main__:PyCaret experiment successfully initialized
